# 🎯 Kuggle × LLM 실습 — 5분이면 끝 (Gemini 버전)

**LLM 활용과 AI 자동화의 기초** · M 증권사 상품솔루션팀 멘토 세션

---

## 오늘 5분 동안 만들 것
1. **STEP 1**: 환경 세팅 (`pip install google-genai` + API Key 등록)
2. **STEP 2**: Gemini API 한 번 호출하기
3. **STEP 3**: 프롬프트 엔지니어링 — `system_instruction` 한 줄의 위력
4. **STEP 4**: ML × LLM 미니 결합 — 클러스터 자동 네이밍

## 시작 전 준비
- (필수) Google 계정으로 코랩 로그인
- (필수) **Google AI Studio**에서 API Key 발급 → https://aistudio.google.com  
  *카드 등록 필요 없음 · 즉시 무료 사용 가능*
- (권장) 우측 상단 `Copy to Drive` 눌러 **내 드라이브로 사본 저장** → 자유롭게 수정

## 왜 Gemini?
Google AI Studio는 **카드 등록 없이 무료**로 즉시 사용할 수 있어 30분 세션에 가장 적합합니다.  
Claude API(Anthropic) 사용을 원하면 [README](./README.md)의 부록을 참고하세요. 함수 시그니처는 거의 동일합니다.

---

## STEP 1 — 환경 세팅

Google의 신규 공식 SDK(`google-genai`)만 설치하면 됩니다.

In [ ]:
!pip install -q google-genai

### API Key 등록 — Colab Secrets 사용 (권장)

**① 키 발급**  
https://aistudio.google.com → 좌측 `Get API key` → `Create API key` → 발급된 키 복사

**② Colab에 등록**  
좌측 사이드바 🔑 **자물쇠 아이콘** → `+ Add new secret`
- Name: `GOOGLE_API_KEY`
- Value: 복사한 키 (`AIza...`)
- **Notebook access 토글 ON**

*키를 노트북 안에 절대 하드코딩하지 마세요. GitHub에 그대로 올라갑니다.*

In [ ]:
import os
from google.colab import userdata
from google import genai
from google.genai import types

os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")

client = genai.Client()
print("✅ Gemini client ready")

## STEP 2 — 첫 호출

`client.models.generate_content()` 한 번이면 끝. 코드 5줄 약속 지킵니다.

In [ ]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="ETF가 뭔지 통계학과 2학년에게 한 줄로 설명해줘.",
)

print(response.text)

### 함수로 감싸기 — 이제부터 LLM이 "함수"가 됩니다

In [ ]:
def ask(prompt: str, system: str = "한 줄로 답하라.", temperature: float = 0.2) -> str:
    """Gemini에게 한 번 묻고 답 텍스트만 돌려준다."""
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=system,
            temperature=temperature,
            max_output_tokens=512,
        ),
    )
    return response.text


print(ask("분산투자의 핵심 아이디어를 1문장으로."))

## STEP 3 — 프롬프트 엔지니어링: `system_instruction` 한 줄이 모든 걸 바꾼다

같은 질문 → 다른 페르소나 두 번 호출해서 답이 어떻게 달라지는지 확인.

In [ ]:
q = "레버리지 ETF, 사도 될까요?"

print("🧑\u200d🏫 [통계학과 교수 모드]")
print(ask(q, system="너는 보수적인 통계학과 교수다. 확률·기댓값 관점에서 답하라."))

print("\n" + "-" * 60 + "\n")

print("💼 [증권사 ETF 애널리스트 모드]")
print(ask(q, system="너는 증권사 ETF 애널리스트다. 수치를 단정하지 말고 [확인 필요] 태그를 붙여라."))

### Structured Output — JSON으로 받아 후처리 자동화

방법 1: `system_instruction`에 "JSON만 출력"을 명시 + `json.loads`로 파싱.  
방법 2 (고급): `response_mime_type="application/json"`을 설정하면 모델이 JSON으로만 답하도록 강제됩니다.

In [ ]:
import json

# 방법 1 — system_instruction 강제
raw = ask(
    prompt="질문: '레버리지 ETF 추천해줘' — 이 의도를 분류하라.",
    system=(
        "너는 검색 의도 분류기다. 아래 JSON 형식으로만 답하라. 다른 텍스트 절대 금지.\n"
        '{"intent": "informational | commercial | transactional", '
        '"confidence": 0.0~1.0, "reason": "한 줄"}'
    ),
    temperature=0.0,
)
print("raw:", raw)

parsed = json.loads(raw)
print("\nparsed type:", type(parsed))
print("intent:", parsed["intent"])
print("confidence:", parsed["confidence"])

In [ ]:
# 방법 2 — response_mime_type으로 JSON 응답 강제 (Gemini 전용 기능)
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="질문: '미국 S&P500 ETF 사고싶다' — intent / confidence / reason 필드를 가진 JSON 객체로 답하라.",
    config=types.GenerateContentConfig(
        temperature=0.0,
        response_mime_type="application/json",
    ),
)
print(json.loads(response.text))

## STEP 4 — ML × LLM 미니 결합

ETF 트렌드 파이프라인의 핵심 아이디어 재현:  
ML이 만든 **키워드 클러스터** → LLM이 **사람이 읽을 수 있는 이름**을 붙인다.

In [ ]:
# 실제 파이프라인에선 TF-IDF + K-Means로 자동 생성된 결과
clusters = {
    "c1": ["비트코인", "달러", "기관", "유입", "현물", "ETF"],
    "c2": ["레버리지", "매수", "손실", "심리", "변동", "손절"],
    "c3": ["자녀", "증여", "세금", "계좌", "미성년", "연금"],
    "c4": ["반도체", "AI", "전력", "인프라", "실적", "엔비디아"],
}

system_prompt = (
    "너는 ETF 콘텐츠 마케터다. "
    "키워드 묶음을 보고 한국어 2~4단어로 클러스터를 명명하라. "
    "JSON만 출력. 형식: {\"name\": \"...\", \"reason\": \"...\"}"
)

for cid, keywords in clusters.items():
    raw = ask(
        prompt=f"키워드: {keywords}",
        system=system_prompt,
        temperature=0.3,
    )
    info = json.loads(raw)
    print(f"{cid:>3}  →  {info['name']:<20}  ({info['reason']})")

**🎉 예상 출력**
```
c1  →  디지털자산 기관유입       (비트코인 현물 ETF 자금 유입 흐름)
c2  →  레버리지 매수심리         (변동성 + 손실 + 행동편향)
c3  →  자녀계좌·증여 절세        (미성년 계좌 + 증여세 설계)
c4  →  AI 반도체·전력 인프라     (AI 데이터센터 + 전력 수요)
```

👉 ML이 *무엇이* 묶이는지 찾고, LLM이 *왜 묶이는지·뭐라 부를지* 알려주는 패턴.  
이게 그대로 "증권사 콘텐츠 자동화 파이프라인"의 가장 비싼 한 칸입니다.

---

## 🚀 여기서부터는 혼자 해보기

1. **자기 분야로 바꾸기** — `clusters`에 본인 학과·관심사 키워드 묶음을 넣어보기 (스포츠, 게임, 음악 등).
2. **출력 포맷 확장** — JSON 스키마에 `target_audience`, `suggested_title` 등 필드 추가.
3. **모델 비교** — `gemini-2.5-flash` ↔ `gemini-2.5-pro`로 바꿔 품질·속도·비용 차이 체감.
4. **Few-shot** — `contents`에 "예시 입력→출력" 쌍을 미리 넣어 답 톤 고정해보기.
5. **Search Grounding** — Gemini의 `tools=[types.Tool(google_search=...)]` 옵션으로 실시간 웹검색 결합.

## 📚 다음 학습 리소스
- Gemini API Docs: https://ai.google.dev/gemini-api/docs
- Google AI Studio (Playground): https://aistudio.google.com
- Gemini Cookbook (예제): https://github.com/google-gemini/cookbook
- Pinecone Learn (벡터 DB·RAG): https://www.pinecone.io/learn/
- M 증권사 ETF 파이프라인 예시 결과물: [examples/](./examples/) — 세션 슬라이드 Slide 13~17 참고

## 💡 부록 — Anthropic Claude로 동일 코드 작성하기
함수 시그니처는 거의 같습니다. README 부록을 참고하거나 아래처럼 바꾸면 됩니다:
```python
# pip install anthropic
import anthropic
client = anthropic.Anthropic()  # ANTHROPIC_API_KEY 환경변수

def ask(prompt, system="한 줄로 답하라.", temperature=0.2):
    r = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=512,
        temperature=temperature,
        system=system,
        messages=[{"role": "user", "content": prompt}],
    )
    return r.content[0].text
```

## 💬 멘토 컨택
[mentor-email] · 진로·실무·논문 무엇이든 환영합니다.